<p><font size="6" color='grey'> <b>
Machine Learning
</b></font> </br></p>
<p><font size="5" color='grey'> <b>
ROC-AUC - Threshold
</b></font> </br></p>

---


# 0  | Install & Import
***

In [ ]:
# Install

In [ ]:
# Import
import numpy as np
from pandas import read_csv, DataFrame

from sklearn.model_selection import train_test_split
from sklearn.neural_network import MLPClassifier
from sklearn.preprocessing import MinMaxScaler, StandardScaler
from sklearn.metrics import (
    accuracy_score,
    confusion_matrix,
    ConfusionMatrixDisplay,
    classification_report,
    roc_auc_score,
    roc_curve,
    RocCurveDisplay
)

import plotly.express as px
import plotly.subplots as sp
import matplotlib.pyplot as plt

In [ ]:
# Warnung ausstellen
import warnings

warnings.filterwarnings("ignore")

# 1 | Understand
---


<p><font color='black' size="5">📋 Checkliste</font></p>

✅ Aufgabe verstehen</br>
✅ Daten sammeln</br>
✅ Statistische Analyse (Min, Max, Mean, Korrelation, ...)</br>
✅ Datenvisualisierung (Streudiagramm, Box-Plot, ...)</br>
✅ Prepare Schritte festlegen</br>

<p><font color='black' size="5">
📒 Anwendungsfall
</font></p>




**Beschreibung:**   
Diese Arbeit entstand aus dem Wunsch, Gewebeproben ausschließlich auf der Grundlage einer Feinnadelpunktion (FNA) genau zu diagnostizieren. In Zusammenarbeit mit Prof. Mangasarian und zwei seiner Doktoranden, Rudy Setiono und Kristin Bennett , wurde mithilfe der Multisurface-Methode (MSM) zur Mustertrennung dieser neun Merkmale ein Klassifikator erstellt, der 97 % der neuen Fälle erfolgreich diagnostizierte. Der resultierende Datensatz ist als Wisconsin Breast Cancer Data bekannt.


Die Arbeit an der Bildanalyse begann 1990 mit der Aufnahme von Nick Street in das Forschungsteam. Ziel war es, die Probe anhand eines digitalen Bildes eines kleinen Abschnitts des FNA-Objektträgers zu diagnostizieren.

**Diagnoseablauf:**

Es wird aus dem Gewebe eine FNA entnommen. Dieses Material wird dann auf einen Objektträger montiert und gefärbt, um die Zellkerne hervorzuheben. Ein Teil des Objektträgers, in dem die Zellen gut differenziert sind, wird dann mit einer Digitalkamera und einem Framegrabber-Board gescannt.
Anschließend isoliert der Anwender die einzelnen Zellkerne . Mit einem Mauszeiger zeichnet der Benutzer die ungefähre Grenze jedes Kerns. Mithilfe eines Computer-Vision-Ansatzes, konvergieren diese Annäherungen dann an die genauen nuklearen Grenzen. Sobald alle (oder die meisten) Kerne auf diese Weise isoliert wurden, berechnet das Programm Werte für jedes der zehn Merkmale jedes Kerns und misst Größe, Form und Textur. Der Mittelwert, der Standardfehler und die Extremwerte dieser Merkmale werden berechnet, was zu insgesamt 30 Kernmerkmalen für jede Probe führt.

[DataSet](https://archive.ics.uci.edu/dataset/15/breast+cancer+wisconsin+original)

[Info](https://pages.cs.wisc.edu/~olvi/uwmp/cancer.html)

**Features:**


+ Dicke: 1 - 10
+ Einheitlichkeit der Zellgröße: 1 - 10
+ Gleichmäßigkeit der Zellform: 1 - 10
+ Randhaftung: 1 - 10
+ Größe einzelner Epithelzellen: 1 - 10
+ Nackte Kerne: 1 - 10
+ Blandes Chromatin: 1 - 1
+ Normale Nukleolen: 1 - 10
+ Mitosen: 1 - 10

**Klassen:**

+ Klasse: (2 für gutartig, 4 für bösartig)

In [ ]:
import pandas as pd
df = read_csv('https://raw.githubusercontent.com/ralf-42/ML_Intro/main/02_daten/05_tabellen/breast_cancer_wisconsin.csv')

In [ ]:
data = df.copy()
target = data.pop("Class")

# 2 | Prepare

---

<p><font color='black' size="5">📋 Checkliste</font></p>

✅ Datentyp ermitteln/ändern</br>
✅ Train-Test-Split durchführen</br>
✅ Nicht benötigte Features löschen</br>
✅ Missing Values behandeln</br>
✅ Ausreißer behandeln</br>
✅ Kategorischer Features Kodieren</br>
✅ Numerischer Features skalieren</br>
✅ Feature-Engineering (neue Features schaffen)</br>
✅ Dimensionalität reduzieren</br>
✅ Resampling (Over-/Undersampling)</br>
✅ Pipeline erstellen/konfigurieren</br>

<font color='black' size="5">
Datentyp ermitteln
</font>

In [ ]:
all_col = data.columns
num_col = data.select_dtypes(include="number").columns
cat_col = data.select_dtypes(exclude="number").columns

<p><font color='black' size="5">
Kodieren Target auf 0/1
</font></p>

In [ ]:
target.replace([2, 4], [0, 1], inplace=True)

<font color='black' size="5">
Missing Values löschen
</font>

In [ ]:
data = data.dropna()
target = target.loc[data.index]

<font color='black' size="5">
Train-Test-Set
</font>

In [ ]:
data_train, data_test, target_train, target_test = train_test_split(
    data, target, test_size=0.20, stratify=target, random_state=42)

<font color='black' size="5">
Skalierung
</font>

In [ ]:
scaler = StandardScaler()
scaler.fit(data_train[all_col])
data_train[all_col] = scaler.transform(data_train[all_col])
data_test[all_col] = scaler.transform(data_test[all_col])

# 3 | Modeling
---

<p><font color='black' size="5">📋 Checkliste</font></p>

✅ Modellauswahl treffen</br>
✅ Pipeline erweitern/konfigurieren</br>
✅ Training durchführen</br>
✅ Hyperparameter Tuning</br>
✅ Cross-Valdiation</br>
✅ Bootstrapping</br>
✅ Regularization</br>

<p><font color='black' size="5">🧠 Algorithmus: MLP – Mehrschichtiges Perzeptron (Klassifikation)</font></p>

Ein MLP (Multi-Layer Perceptron) ist ein künstliches neuronales Netz mit mehreren Schichten. Jede Schicht besteht aus Neuronen, die gewichtete Eingaben empfangen, eine Aktivierungsfunktion anwenden und das Ergebnis an die nächste Schicht weitergeben. Am Ende steht eine Klassenentscheidung.

**Analogie:** Wie in einer Fabrik werden Informationen von Station zu Station weitergegeben und schrittweise verfeinert – bis eine Entscheidung steht.

**Wichtige Parameter:**

| Parameter | Bedeutung |
|-----------|----------|
| `hidden_layer_sizes` | Anzahl und Größe der versteckten Schichten |
| `activation` | Aktivierungsfunktion (z.B. `relu`, `tanh`) |
| `max_iter` | Maximale Anzahl Trainings-Epochen |
| `random_state` | Reproduzierbarkeit |

In [ ]:
#@markdown   <p><font size="4" color='green'>  MLP – Netzwerkstruktur</font> </br></p>

import base64
from IPython.display import Image, display

diagram = """
flowchart LR
    IN[Eingabeschicht] --> H1[Versteckte Schicht 1]
    H1 --> H2[Versteckte Schicht 2]
    H2 --> OUT[Ausgabeschicht - Klasse]
"""

encoded = base64.urlsafe_b64encode(diagram.strip().encode()).decode()
display(Image(url=f"https://mermaid.ink/img/{encoded}", width=650))

 <p><font color='black' size="5">
Modellauswahl & Training
</font></p>

In [ ]:
model = MLPClassifier(verbose=0, random_state=42)

In [ ]:
model.fit(data_train, target_train)

<p><font color='black' size="5">
Loss-Entwicklung
</font></p>

In [ ]:
title_ = "Loss-Entwicklung"
px.line(
    y=model.loss_curve_,
    title=title_,
    labels={"x": "Epochen", "y": "Loss-Wert"},
    width=800,
    height=400,
)

# 4 | Evaluate
---

<p><font color='black' size="5">📋 Checkliste</font></p>

✅ Prognose (Train, Test) erstellen</br>
✅ Modellgüte prüfen</br>
✅ Residuenanalyse erstellen</br>
✅ Feature Importance/Selektion prüfen</br>
✅ Robustheitstest erstellen</br>
✅ Modellinterpretation erstellen</br>
✅ Sensitivitätsanalyse erstellen</br>
✅ Kommunikation (Key Takeaways)</br>

<p><font color='black' size="5">
Prognose
</font></p>

In [ ]:
# Prognose 0/1
target_train_pred = model.predict(data_train)
target_test_pred = model.predict(data_test)

In [ ]:
# Prognose Wahrscheinlichkeiten für 0/1
target_train_proba = model.predict_proba(data_train)
target_test_proba = model.predict_proba(data_test)


<p><font color='black' size="5">
Accuracy
</font></p>

In [ ]:
acc_train = accuracy_score(target_train, target_train_pred) * 100
print(f"Modell: {model} -- Train -- Accuracy: {acc_train:5.2f}")

In [ ]:
acc_test = accuracy_score(target_test, target_test_pred) * 100
print(f"Modell: {model} -- Test -- Accuracy: {acc_test:5.2f}")


<p><font color='black' size="5">
Confusion Matrix
</font></p>

In [ ]:
display_labels = ["Negativ", "Positiv"]
conf_matrix = confusion_matrix(target_test, target_test_pred)
disp = ConfusionMatrixDisplay(conf_matrix, display_labels=display_labels)
disp.plot(cmap="Blues")

In [ ]:
print(classification_report(target_test, target_test_pred, target_names=display_labels))

<p><font color='black' size="5">
ROC-AUC
</font></p>

Die ROC-AUC Analyse ist ein Werkzeug, um beim maschinellen Lernen die Leistung eines Klassifikationsmodells zu bewerten.  

1. **ROC-Kurve (Receiver Operating Characteristic)**: Dies ist ein Graph, der zeigt, wie gut ein Modell zwischen zwei Klassen (zum Beispiel positiv und negativ) unterscheiden kann. Auf der x-Achse wird die Falsch-Positiv-Rate (FPR) abgebildet, und auf der y-Achse die Wahr-Positiv-Rate (TPR). Die FPR zeigt, wie oft das Modell fälschlicherweise eine negative Instanz als positiv klassifiziert, während die TPR zeigt, wie oft das Modell korrekt eine positive Instanz als positiv erkennt.

$$
\text{TPR} = \frac{\text{TP}}{\text{TP} + \text{FN}}
$$
<br>

$$
\text{FPR} = \frac{\text{FP}}{\text{FP} + \text{TN}}
$$
<br>

2. **AUC (Area Under the Curve)**: Dies ist ein Maß dafür, wie gut das Modell insgesamt ist. Es wird durch Berechnung der Fläche unter der ROC-Kurve ermittelt. Ein AUC-Wert von 1,0 bedeutet, dass das Modell perfekt klassifiziert, während ein Wert von 0,5 bedeutet, dass es nicht besser ist als zufälliges Raten.


Die ROC-Kurve wird durch Variation der Schwellwerte für die Entscheidungsgrenze des Modells zwischen 0 und 1 ermittelt.

- **Schwellwert (Threshold)**: Dies ist der Wert, bei dem das Modell entscheidet, ob eine Instanz als positiv oder negativ klassifiziert wird. Indem man den Schwellwert von 0 bis 1 variiert, verändert man, wann das Modell eine Instanz als positiv klassifiziert.

- **Variation der Schwellwerte**: Bei einem niedrigen Schwellwert klassifiziert das Modell mehr Instanzen als positiv, was zu einer höheren Wahr-Positiv-Rate (TPR) und möglicherweise auch zu einer höheren Falsch-Positiv-Rate (FPR) führt. Umgekehrt führt ein höherer Schwellwert dazu, dass das Modell weniger Instanzen als positiv klassifiziert, was die FPR senken kann, aber auch die TPR verringert.

- **Erstellung der ROC-Kurve**: Für jeden Schwellwert wird ein Punkt auf der ROC-Kurve gezeichnet, der aus der FPR und TPR besteht. Die Kurve zeigt, wie sich die TPR und FPR verändern, wenn der Schwellwert variiert wird.


<p><font color='black' size="5">
ROC-AUC Kurve (Yellowbrick)
</font></p>

In [ ]:
from yellowbrick.classifier import ROCAUC

visualizer = ROCAUC(model, micro=False, macro=False, classes=["Negativ", "Positiv"])

visualizer.fit(data_train, target_train)  # Fit the training data to the visualizer
visualizer.score(data_test, target_test)  # Evaluate the model on the test data
visualizer.show()

<p><font color='black' size="5">
ROC-AUC Kurve sklearn
</font></p>

In [ ]:
RocCurveDisplay.from_predictions(
    target_test, target_test_proba[:, 1], name="MLPClassifier"
)
plt.plot([0, 1], [0, 1], linestyle="--", color="red", label="Random")

plt.legend()
plt.show()

In [ ]:
rocauc = roc_auc_score(target_train, model.predict_proba(data_train)[:, 1])
print(f"AUC Train: {rocauc:.2f}")

rocauc = roc_auc_score(target_test, model.predict_proba(data_test)[:, 1])
print(f"AUC Test: {rocauc:.2f}")

<p><font color='black' size="5">
Optimaler Schwellenwert
</font></p>

 Der Punkt auf der Kurve, der der linken oberen Ecke am nächsten liegt.

In [ ]:
fpr, tpr, thresholds = roc_curve(target_test, target_test_proba[:, 1])

In [ ]:
# Euklidischen Abstand zu jedem Punkt von der linken oberen Ecke berechnen
distances = np.sqrt((1 - tpr) ** 2 + fpr**2)

# Den Index des minimalen Abstands finden
optimal_idx = np.argmin(distances)

# Den optimalen Schwellenwert finden
optimal_threshold = thresholds[optimal_idx]

print(f"Optimaler Schwellenwert nach Euklidischen Abstand : {optimal_threshold:0.3f}")

In [ ]:
# Manhattan Abstand zu jedem Punkt von der linken oberen Ecke berechnen
manhattan_distances = np.abs(1 - tpr) + np.abs(fpr)

# Den Index des minimalen Abstands finden
optimal_idx = np.argmin(manhattan_distances)

# Den optimalen Schwellenwert finden
optimal_threshold = thresholds[optimal_idx]

print(f"Optimaler Schwellenwert nach Manhatten Abstand: {optimal_threshold:0.3f}")

<p><font color='black' size="5">
Anpassung Schwellwert auf Basis ROC/AUC
</font></p>

In [ ]:
threshold = 0.323  # Distanz: Manhatten und Youden's J-Index
new_target_pred = (target_test_proba[:, 1] >= threshold).astype(int)
conf_matrix = confusion_matrix(target_test, new_target_pred)
disp = ConfusionMatrixDisplay(conf_matrix, display_labels=["No", "Yes"])
disp.plot(cmap="Blues")

In [ ]:
print(classification_report(target_test, new_target_pred, target_names=["No", "Yes"]))

# 5 | Deploy
---

<p><font color='black' size="5">📋 Checkliste</font></p>

✅ Modellexport und -speicherung</br>
✅ Abhängigkeiten und Umgebung</br>
✅ Sicherheit und Datenschutz</br>
✅ In die Produktion integrieren</br>
✅ Tests und Validierung</br>
✅ Dokumentation & Wartung</br>